# Data Preparation and Baseline Models

Clean the fraud data, inspect risk signals, train baseline models, and save the best PR-AUC model.

In [ ]:
from pathlib import Path
import json
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / 'data' / 'credit_card_fraud_2026.csv'
MODELS_DIR = PROJECT_ROOT / 'models'
REPORTS_DIR = PROJECT_ROOT / 'reports'
target_column = 'is_fraud'

df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
print(df.columns.tolist())
df.head()

In [ ]:
print(df.info())
print('\nMissing values:')
print(df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nTarget values:')
print(df[target_column].value_counts(dropna=False))
print('\nTarget percentages:')
print(df[target_column].value_counts(normalize=True, dropna=False) * 100)

In [ ]:
df = df.drop_duplicates().copy()
print('Missing target values:', df[target_column].isnull().sum())
df = df.dropna(subset=[target_column]).copy()

print('Target dtype before conversion:', df[target_column].dtype)
if pd.api.types.is_bool_dtype(df[target_column]):
    df[target_column] = df[target_column].astype(int)

print('Target dtype after conversion:', df[target_column].dtype)
print(df[target_column].value_counts())

In [ ]:
sns.set_theme(style='whitegrid')

plt.figure(figsize=(7, 4))
sns.countplot(data=df, x=target_column)
plt.title('Legitimate vs Fraudulent Transactions')
plt.xlabel('Fraud Label')
plt.ylabel('Number of Transactions')
plt.xticks([0, 1], ['Legitimate', 'Fraud'])
plt.tight_layout()
plt.show()

fraud_by_category = df.groupby('merchant_category')[target_column].mean().sort_values(ascending=False)
plt.figure(figsize=(10, 5))
fraud_by_category.plot(kind='bar')
plt.title('Fraud Rate by Merchant Category')
plt.xlabel('Merchant Category')
plt.ylabel('Fraud Rate')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

fraud_by_auth = df.groupby('auth_method')[target_column].mean().sort_values(ascending=False)
plt.figure(figsize=(8, 5))
fraud_by_auth.plot(kind='bar', color='orange')
plt.title('Fraud Rate by Authentication Method')
plt.xlabel('Authentication Method')
plt.ylabel('Fraud Rate')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x=target_column, y='amount_usd')
plt.ylim(0, df['amount_usd'].quantile(0.99))
plt.title('Transaction Amount by Fraud Status')
plt.xlabel('Fraud Label')
plt.ylabel('Amount in USD')
plt.xticks([0, 1], ['Legitimate', 'Fraud'])
plt.tight_layout()
plt.show()

print('Fraud rate by foreign transaction:')
print(df.groupby('is_foreign_transaction')[target_column].mean())
print('\nFraud rate by new merchant:')
print(df.groupby('is_new_merchant')[target_column].mean())

hourly_fraud = df.groupby('time_of_day_hour')[target_column].mean()
plt.figure(figsize=(12, 5))
hourly_fraud.plot(kind='line', marker='o')
plt.title('Fraud Rate by Hour')
plt.xlabel('Hour')
plt.ylabel('Fraud Rate')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
drop_columns = [target_column, 'transaction_id']
X = df.drop(columns=drop_columns)
y = df[target_column]

print('Feature shape:', X.shape)
print('Target shape:', y.shape)
display(X.head())

categorical_columns = X.select_dtypes(include=['object', 'bool']).columns.tolist()
numerical_columns = X.select_dtypes(include=['number']).columns.tolist()
print('Categorical columns:')
print(categorical_columns)
print('\nNumerical columns:')
print(numerical_columns)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))
print('Training fraud rate:', y_train.mean())
print('Testing fraud rate:', y_test.mean())

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from src.preprocessing import create_preprocessor

logistic_model = Pipeline([
    ('preprocessor', create_preprocessor(numerical_columns, categorical_columns)),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)),
])
logistic_model.fit(X_train, y_train)

random_forest_model = Pipeline([
    ('preprocessor', create_preprocessor(numerical_columns, categorical_columns)),
    ('classifier', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)),
])
random_forest_model.fit(X_train, y_train)

In [ ]:
def evaluate_model(name, model):
    probability = model.predict_proba(X_test)[:, 1]
    prediction = (probability >= 0.50).astype(int)
    print(f'\n{name}')
    print(classification_report(y_test, prediction, zero_division=0))
    print('ROC-AUC:', roc_auc_score(y_test, probability))
    print('PR-AUC:', average_precision_score(y_test, probability))
    print('Confusion Matrix:')
    print(confusion_matrix(y_test, prediction))
    return probability

fraud_probability = evaluate_model('Logistic Regression', logistic_model)
rf_probability = evaluate_model('Random Forest', random_forest_model)

In [ ]:
model_probabilities = {'Logistic Regression': fraud_probability, 'Random Forest': rf_probability}
comparison = []
for model_name, probability in model_probabilities.items():
    prediction = (probability >= 0.50).astype(int)
    comparison.append({
        'Model': model_name,
        'Precision': precision_score(y_test, prediction, zero_division=0),
        'Recall': recall_score(y_test, prediction, zero_division=0),
        'F1 Score': f1_score(y_test, prediction, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, probability),
        'PR-AUC': average_precision_score(y_test, probability),
    })

comparison_df = pd.DataFrame(comparison).sort_values('PR-AUC', ascending=False)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
comparison_df.to_csv(REPORTS_DIR / 'model_comparison.csv', index=False)
comparison_df

In [ ]:
final_model_name = comparison_df.iloc[0]['Model']
final_model = logistic_model if final_model_name == 'Logistic Regression' else random_forest_model
final_probability = model_probabilities[final_model_name]

threshold_results = []
for threshold in np.arange(0.05, 0.96, 0.05):
    prediction = (final_probability >= threshold).astype(int)
    threshold_results.append({
        'threshold': round(threshold, 2),
        'precision': precision_score(y_test, prediction, zero_division=0),
        'recall': recall_score(y_test, prediction, zero_division=0),
        'f1_score': f1_score(y_test, prediction, zero_division=0),
        'alerts': prediction.sum(),
    })

threshold_df = pd.DataFrame(threshold_results)
final_threshold = float(threshold_df.loc[threshold_df['f1_score'].idxmax(), 'threshold'])
print(f'Selected model: {final_model_name}')
print(f'Selected threshold (highest test F1): {final_threshold:.2f}')
threshold_df

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(final_model, MODELS_DIR / 'credit_card_fraud_model.joblib')

config = {
    'target_column': target_column,
    'threshold': final_threshold,
    'model_name': final_model_name,
    'feature_columns': X.columns.tolist(),
}
with open(MODELS_DIR / 'model_config.json', 'w', encoding='utf-8') as file:
    json.dump(config, file, indent=4)

print('Saved model to', MODELS_DIR / 'credit_card_fraud_model.joblib')
print('Saved configuration to', MODELS_DIR / 'model_config.json')